# GNU Radio Conference 2026 CTF: Decode the Airwaves

Five SigMF recordings contain five different signal classes. Use the released TorchSig Models XCiT classifier to identify each recording, then concatenate the first character of each predicted class in capture order.

## 1. Imports and paths

The checkpoint is downloaded from the official TorchSig Models v1.0.0 release if it is not already present.

In [ ]:
import json
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import torch
from torchsig_models.models import XCiTClassifier

HERE = Path.cwd()
if not (HERE / 'captures').exists():
    HERE = HERE / 'examples' / 'ctf_xcit_sigmf'
CAPTURE_DIR = HERE / 'captures'
CHECKPOINT = HERE / 'xcit_narrowband_v1.0.0.ckpt'
CHECKPOINT_URL = (
    'https://github.com/TorchDSP/torchsig-models/releases/download/'
    'v1.0.0/xcit_narrowband_v1.0.0.ckpt'
)
if not CHECKPOINT.exists():
    print('Downloading the official pretrained XCiT checkpoint...')
    urlretrieve(CHECKPOINT_URL, CHECKPOINT)
print(f'Checkpoint: {CHECKPOINT}')

## 2. Load and inspect the SigMF recordings

SigMF keeps the complex sample stream in `.sigmf-data` and recording metadata in `.sigmf-meta`. TorchSig 2.1.1 does not provide a SigMF file handler, so this challenge reads its `cf32_le` recordings directly with NumPy and uses the metadata to obtain each sample rate.

In [ ]:
def read_sigmf(meta_path):
    metadata = json.loads(meta_path.read_text(encoding='utf-8'))
    datatype = metadata['global']['core:datatype']
    if datatype != 'cf32_le':
        raise ValueError(f'Unsupported SigMF datatype: {datatype}')
    data_path = meta_path.with_suffix('.sigmf-data')
    return np.fromfile(data_path, dtype='<c8'), metadata

meta_paths = sorted(CAPTURE_DIR.glob('capture_*.sigmf-meta'))
recordings = [read_sigmf(path) for path in meta_paths]
iq_samples = np.stack([iq for iq, _metadata in recordings])

fig, axes = plt.subplots(len(iq_samples), 1, figsize=(10, 10), constrained_layout=True)
for path, (iq, metadata), axis in zip(meta_paths, recordings, axes):
    sample_rate = metadata['global']['core:sample_rate']
    axis.specgram(iq, NFFT=256, Fs=sample_rate, noverlap=192)
    axis.set_title(path.stem)
    axis.set_ylabel('Hz')
axes[-1].set_xlabel('Time (s)')
plt.show()

## 3. Restore the pretrained model

The release checkpoint was trained with TorchSig 2.1.1. Its 57 outputs follow the signal-generator registry order below. This historical order must be retained even when the notebook runs with a newer TorchSig installation.

In [ ]:
CLASS_NAMES_V211 = [
    'tone', 'ofdm-64', 'ofdm-72', 'ofdm-128', 'ofdm-180', 'ofdm-256',
    'ofdm-300', 'ofdm-512', 'ofdm-600', 'ofdm-900', 'ofdm-1024',
    'ofdm-1200', 'ofdm-2048', 'lfm-data', 'lfm-radar', '2fsk', '4fsk',
    '8fsk', '16fsk', '2gfsk', '4gfsk', '8gfsk', '16gfsk', '2msk',
    '4msk', '8msk', '16msk', '2gmsk', '4gmsk', '8gmsk', '16gmsk',
    'fm', 'ook', 'bpsk', 'qpsk', '8psk', '16psk', '32psk', '64psk',
    '4ask', '8ask', '16ask', '32ask', '64ask', '16qam', '32qam',
    '64qam', '256qam', '1024qam', '32qam_cross', '128qam_cross',
    '512qam_cross', 'chirpss', 'am-dsb', 'am-dsb-sc', 'am-usb', 'am-lsb',
]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = XCiTClassifier.load_from_checkpoint(CHECKPOINT, map_location=device)
model.to(device).eval()
print(f'Loaded {len(CLASS_NAMES_V211)}-class XCiT on {device}')

## 4. Classify the captures

`ComplexTo2D`, used during TorchSig training, represents complex IQ as two real channels. The equivalent conversion below produces a `[batch, 2, samples]` tensor.

In [ ]:
model_input = torch.from_numpy(
    np.stack([np.stack((iq.real, iq.imag)) for iq in iq_samples])
).to(device=device, dtype=torch.float32)

with torch.inference_mode():
    probabilities = model(model_input).softmax(dim=1).cpu()

predictions = []
for path, scores in zip(meta_paths, probabilities):
    confidence, class_index = scores.max(dim=0)
    class_name = CLASS_NAMES_V211[class_index.item()]
    predictions.append(class_name)
    print(f'{path.stem}: {class_name:10s} ({confidence.item():.1%})')

## 5. Recover the flag

Take the first character of every predicted class, preserving capture order.

In [ ]:
answer = ''.join(class_name[0] for class_name in predictions).upper()
print(f'CTF answer: {answer}')